# Chest CT Training Notebook

Kaggle-ready ResNet18 training notebook with mixed precision support for T4/P100 GPUs.

Set `DATA_DIR` to the Kaggle dataset folder if you use a custom path.

In [ ]:
import copy
import json
import os
import random
from pathlib import Path

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Subset, random_split
from torchvision import datasets, models, transforms

MODEL_NAME = "chest"
DEFAULT_DATA_DIR = "/kaggle/input/chest-ctscan-images"
DATA_DIR = Path(os.getenv("CHEST_DATA_DIR", DEFAULT_DATA_DIR))
MODEL_SAVE_PATH = Path("best_chest_model.pth")
CLASSES_JSON_PATH = Path("classes.json")

BATCH_SIZE = 32
EPOCHS = 12
LEARNING_RATE = 1e-4
VAL_SPLIT = 0.2
SEED = 42
NUM_WORKERS = 2

torch.manual_seed(SEED)
random.seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.benchmark = True

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
pin_memory = device.type == "cuda"
scaler = torch.cuda.amp.GradScaler(enabled=pin_memory)

print(f"Model: {MODEL_NAME}")
print(f"Device: {device}")
print(f"Data dir: {DATA_DIR}")

if not DATA_DIR.exists():
    raise FileNotFoundError(f"Dataset directory not found: {DATA_DIR}")

train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

train_path = DATA_DIR / "train"
val_path = DATA_DIR / "val"

if train_path.is_dir() and val_path.is_dir():
    train_dataset = datasets.ImageFolder(str(train_path), transform=train_transform)
    val_dataset = datasets.ImageFolder(str(val_path), transform=val_transform)
    class_names = train_dataset.classes
else:
    full_dataset = datasets.ImageFolder(str(DATA_DIR))
    class_names = full_dataset.classes
    if len(full_dataset) < 2:
        raise ValueError("Dataset must contain at least two images to create a validation split.")

    val_size = max(1, int(VAL_SPLIT * len(full_dataset)))
    val_size = min(val_size, len(full_dataset) - 1)
    train_size = len(full_dataset) - val_size

    indices = list(range(len(full_dataset)))
    train_indices, val_indices = random_split(indices, [train_size, val_size], generator=torch.Generator().manual_seed(SEED))

    train_dataset = Subset(datasets.ImageFolder(str(DATA_DIR), transform=train_transform), train_indices.indices)
    val_dataset = Subset(datasets.ImageFolder(str(DATA_DIR), transform=val_transform), val_indices.indices)

CLASSES_JSON_PATH.write_text(json.dumps(class_names, indent=2), encoding="utf-8")
print(f"Classes: {class_names}")

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=pin_memory,
)
val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=pin_memory,
)

def build_model(num_classes):
    try:
        model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
        print("Loaded ImageNet pretrained weights.")
    except Exception as exc:
        print(f"Falling back to random initialization: {exc}")
        model = models.resnet18(weights=None)
    model.fc = nn.Linear(model.fc.in_features, num_classes)
    return model.to(device)

def train_model():
    model = build_model(len(class_names))
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

    best_model_wts = copy.deepcopy(model.state_dict())
    best_acc = 0.0

    for epoch in range(EPOCHS):
        print(f"Epoch {epoch + 1}/{EPOCHS}")
        print("-" * 10)

        for phase, loader in (("train", train_loader), ("val", val_loader)):
            model.train() if phase == "train" else model.eval()
            running_loss = 0.0
            running_corrects = 0

            for inputs, labels in loader:
                inputs = inputs.to(device, non_blocking=pin_memory)
                labels = labels.to(device, non_blocking=pin_memory)
                optimizer.zero_grad(set_to_none=True)

                with torch.set_grad_enabled(phase == "train"):
                    with torch.cuda.amp.autocast(enabled=pin_memory):
                        outputs = model(inputs)
                        loss = criterion(outputs, labels)
                        _, preds = torch.max(outputs, 1)

                    if phase == "train":
                        scaler.scale(loss).backward()
                        scaler.step(optimizer)
                        scaler.update()

                running_loss += loss.item() * inputs.size(0)
                running_corrects += torch.sum(preds == labels.data)

            epoch_loss = running_loss / len(loader.dataset)
            epoch_acc = running_corrects.double() / len(loader.dataset)
            print(f"{phase} Loss: {epoch_loss:.4f} Acc: {epoch_acc:.4f}")

            if phase == "val" and epoch_acc > best_acc:
                best_acc = epoch_acc
                best_model_wts = copy.deepcopy(model.state_dict())
                torch.save(model.state_dict(), MODEL_SAVE_PATH)
                print(f"Saved best checkpoint to {MODEL_SAVE_PATH}")

    print(f"Best val Acc: {best_acc:.4f}")
    model.load_state_dict(best_model_wts)
    return model

trained_model = train_model()
trained_model.eval()
